# Information retrieval: building a search engine by hand

## What is information retrieval?

Finding, in a collection of documents, the ones that best answer a question. There are only three ideas in it:

* **documents**: the things we search through;
* a **query**: what the user is looking for;
* **relevance**: a score that says how well a document answers the query.

The whole of this notebook is about that third word. Everything we need we have already learnt this week:

| we need to... | and we already know |
|---|---|
| cut a text into words | `.split()` and `.lower()` |
| clean up the punctuation | `re.sub()`|
| count the words | `Counter` |
| keep counts per document | dictionaries  |
| return the best documents | sorting — the one new thing here |

Tomorrow we will do exactly this over ten thousand books, in about three lines of `pandas`. Doing it by hand once means you will know what those three lines are doing.

## Our corpus

Six documents. In a real project this would be thousands of files; here it is six famous opening lines, short enough that we can check every result by eye — which is going to matter, because our scores are about to become a lot less obvious than "how many times does this word occur".

In [ ]:
documents = {
    "two_cities": "It was the best of times, it was the worst of times, it was the age of wisdom, it was the age of foolishness, it was the epoch of belief, it was the epoch of incredulity, it was the season of Light, it was the season of Darkness.",
    "pride": "It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.",
    "alice": "Alice was beginning to get very tired of sitting by her sister on the bank, and of having nothing to do: once or twice she had peeped into the book her sister was reading, but it had no pictures or conversations in it.",
    "moby_dick": "Call me Ishmael. Some years ago, never mind how long precisely, having little or no money in my purse, and nothing particular to interest me on shore, I thought I would sail about a little and see the watery part of the world.",
    "dalloway": "Mrs Dalloway said she would buy the flowers herself. For Lucy had her work cut out for her.",
    "frankenstein": "You will rejoice to hear that no disaster has accompanied the commencement of an enterprise which you have regarded with such evil forebodings.",
}

print(len(documents), "documents:")
for doc_id in documents:
    print("-", doc_id)

## Step 1: from texts to counts

Before we can count words, we have to cut the text into them. Two problems we have already met this week come back here:

* `The` and `the` are different strings, so they would be counted separately;
* `times,` is not `times`, so punctuation would split one word into two.

Lowercasing fixes the first, and a regular expression that keeps only letters, apostrophes and spaces fixes the second. This operation is called **tokenising**, and the words it produces are called **tokens**.

In [ ]:
import re
from collections import Counter


def tokenise(text):
    """ cut a text into a list of comparable words

    Args:
        text: a string

    Returns:
        A list of lowercased words, without punctuation
    """
    text = text.lower()
    text = re.sub(r"[^a-z' ]", " ", text)
    return text.split()


print(tokenise(documents["two_cities"]))

Now we count the words of every document, and store the counts in a dictionary whose keys are document names and whose values are `Counter` objects.

A dictionary of dictionaries — exactly the shape of the nested JSON we saw in the previous notebook. We keep the length of each document too, because we are going to need it very soon.

In [ ]:
word_counts = dict()
doc_lengths = dict()

for doc_id, text in documents.items():
    tokens = tokenise(text)
    word_counts[doc_id] = Counter(tokens)
    doc_lengths[doc_id] = len(tokens)

print(word_counts["alice"].most_common(5))
print()
print(doc_lengths)

This is the **bag of words**: every document reduced to how often each word occurs in it. The order of the words is gone (that is why it is called a *bag*), and looking a word up is no longer a search through the text but a single lookup:

In [ ]:
print(word_counts["alice"]["sister"])
print(word_counts["alice"]["whale"])

⚠️ Note that a `Counter` returns `0` for a word it has never seen, where a plain dictionary would raise a `KeyError`. That is one of the reasons it is convenient here.

## Step 2: ranking

A search engine does not just find documents, it puts them **in order**. That is all sorting is, and it is the one genuinely new tool in this notebook.

`sorted()` takes a `key=` argument saying *what to sort on*, and `reverse=True` to put the largest first. Applied to the `.items()` of a dictionary it gives back a list of `(name, score)` pairs. Those pairs are **tuples**: like small lists, but immutable — you will meet them constantly as the output of `.items()`, `sorted()` and `.most_common()`.

In [ ]:
example_scores = {"doc_a": 0.2, "doc_b": 0.9, "doc_c": 0.5}

ranked = sorted(example_scores.items(), key=lambda x: x[1], reverse=True)

print(ranked)
print(ranked[0])          # the best document: a (name, score) tuple
print(ranked[:2])         # the best two documents

✏️ **Exercise 1:**

Search the corpus for a single word and print the three best documents, ranked by how often the word occurs.

Suggested steps:
* start from `query = "the"`
* create an empty dictionary called `scores`
* loop over the documents and put `word_counts[doc_id][query]` into `scores`
* sort `scores` from high to low, and print the first three

In [ ]:
query = "the"

# Type your code here:


### The problem: long documents win

<img src="images/long_doc.jpg" width="300" />

Compare your ranking with the document lengths we computed earlier. The winner is simply the longest document in the corpus.

That is not a coincidence and it is not relevance: a long document has more room for any word to occur, so counting raw occurrences rewards length rather than aboutness. Every search engine has to deal with this.

In [ ]:
print(doc_lengths)

✏️ **Exercise 2:**

Fix it: instead of the raw count, use the **proportion** of the document that the query word takes up — the count divided by the length of the document.

This quantity has a name you will see everywhere: **term frequency** (TF).

Then try the query `"had"` twice, with and without dividing by the length. `alice` contains it twice and `dalloway` only once — but `dalloway` is less than half as long. Which of the two is the query really *about*?

In [ ]:
query = "the"

# Type your code here:


✏️ **Exercise 3:**

Now search for a **phrase**: `"of times"`.

⚠️ Notice that `word_counts` cannot help you here. The bag of words threw the word order away, so it cannot tell you whether `of` and `times` were next to each other. For a phrase you have to go back to the original text — and `.count()` from notebook 2a is all you need.

Print how many times the phrase occurs in each document.

In [ ]:
phrase = "of times"

# Type your code here:


✏️ **Exercise 4:**

Real queries have more than one word. The simplest approach: score every word of the query separately, and add the scores up.

Score the corpus against the query `["her", "sister"]`, using term frequency, and print the ranking.

Then look hard at the result. Which document comes first — and is that the document you would want a search engine to give you?

In [ ]:
query = ["her", "sister"]

# Type your code here:


## Step 3: some words are more useful than others

If you did Exercise 4, `dalloway` came first, ahead of `alice`. But `dalloway` only matches `her`, while `alice` matches **both** `her` and `sister` — and `sister` is surely the more informative half of that query.

The reason is that our score treats the two words as equally important. They are not: `the` occurs in five of our six documents, `sister` in exactly one. A word that appears everywhere cannot help us tell documents apart; a rare word is worth a great deal.

So we weight each word by how rare it is in the collection. First we need its **document frequency**: in how many documents does it appear at all? A **set** is the natural tool, because it holds each element only once.

In [ ]:
def document_frequency(word):
    """ in how many documents of the corpus does `word` appear at least once? """
    documents_containing = set()
    for doc_id in word_counts:
        if word_counts[doc_id][word] > 0:
            documents_containing.add(doc_id)
    return len(documents_containing)


for word in ["the", "her", "sister", "money", "whale"]:
    print(word, document_frequency(word))

Then we turn it upside down, so that *rare* words get a *high* weight: divide the number of documents by the document frequency. Taking the logarithm keeps the weight from growing too aggressively.

This is the **inverse document frequency** (IDF).

In [ ]:
import math

number_of_documents = len(documents)


def idf(word):
    """ the inverse document frequency of a word in our corpus:
    high for rare words, low for words that are everywhere
    """
    df = document_frequency(word)
    if df == 0:                # a word that occurs in no document at all
        return 0
    return math.log(number_of_documents / df)


for word in ["the", "her", "sister", "money", "whale"]:
    print(word, round(idf(word), 3))

`the` is worth almost nothing (0.18), `sister` is worth a lot (1.79).

Multiply a word's term frequency in a document by its inverse document frequency across the collection, and you have **TF-IDF** — the scoring scheme search engines were built on for decades, and which you will meet tomorrow as a single line of `scikit-learn`.

✏️ **Exercise 5 (advanced):**

Score the corpus against the query `["her", "sister"]` again, but this time multiply each term frequency by that word's `idf()` before adding it to the document's score.

Does the ranking change?

In [ ]:
query = ["her", "sister"]

# Type your code here:


## Putting it together

Wrapped in a function (notebook 1d), the whole search engine is about ten lines long:

In [ ]:
def search(query, top_k=3):
    """ rank the documents of the corpus against a query, using TF-IDF

    Args:
        query: the search query, as a string
        top_k: how many results to return

    Returns:
        A list of (document, score) tuples, best first
    """
    query_words = tokenise(query)
    scores = dict()

    for doc_id in word_counts:
        score = 0
        for word in query_words:
            term_frequency = word_counts[doc_id][word] / doc_lengths[doc_id]
            score = score + term_frequency * idf(word)
        scores[doc_id] = score

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]


for query in ["her sister", "no money in my purse", "the age of wisdom", "whale"]:
    print(query)
    for doc_id, score in search(query):
        print("   ", doc_id, round(score, 4))

Look at the last query. Nothing in the corpus mentions a whale, so every score is `0` — the engine happily returns three documents anyway. A real search engine would tell the user there are no results. Deciding what a score of zero means is part of the job.

## Where this goes next

You have just built, by hand, the thing that the rest of the week automates:

* **Tomorrow (notebook 3g)**: the same four steps — count, normalise by length, weight by rarity, rank — over a sample of **ten thousand British Library books**, using `pandas` and `scikit-learn`. The whole of this notebook will collapse into a handful of lines. That is the point of `pandas`: not new ideas, but the same ideas at a scale you cannot write out by hand.
* **On Thursday**: the weakness of everything we did here is that it only knows about *words*. A document about a "ship" does not match a query about a "boat", because TF-IDF has no idea the two are related. Word embeddings are the answer to that.

# Solutions

✏️ **Exercise 1:**

In [ ]:
query = "the"

scores = dict()
for doc_id in word_counts:
    scores[doc_id] = word_counts[doc_id][query]

ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

for doc_id, score in ranked[:3]:
    print(doc_id, score)

✏️ **Exercise 2:**

In [ ]:
query = "the"

scores = dict()
for doc_id in word_counts:
    scores[doc_id] = word_counts[doc_id][query] / doc_lengths[doc_id]

ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

for doc_id, score in ranked[:3]:
    print(doc_id, round(score, 4))

In [ ]:
# The query "had": the raw count prefers `alice`, the term frequency prefers `dalloway`,
# which is less than half as long.

query = "had"

for doc_id in word_counts:
    raw_count = word_counts[doc_id][query]
    term_frequency = raw_count / doc_lengths[doc_id]
    print(doc_id, "| length:", doc_lengths[doc_id], "| count:", raw_count,
          "| tf:", round(term_frequency, 4))

✏️ **Exercise 3:**

In [ ]:
phrase = "of times"

for doc_id, text in documents.items():
    print(doc_id, text.lower().count(phrase))

✏️ **Exercise 4:**

In [ ]:
query = ["her", "sister"]

scores = dict()
for doc_id in word_counts:
    score = 0
    for word in query:
        score = score + word_counts[doc_id][word] / doc_lengths[doc_id]
    scores[doc_id] = score

ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

for doc_id, score in ranked[:3]:
    print(doc_id, round(score, 4))

# `dalloway` wins, although it does not contain the word "sister" at all:
# it is short, and it happens to use "her" twice.

✏️ **Exercise 5 (advanced):**

In [ ]:
query = ["her", "sister"]

scores = dict()
for doc_id in word_counts:
    score = 0
    for word in query:
        term_frequency = word_counts[doc_id][word] / doc_lengths[doc_id]
        score = score + term_frequency * idf(word)
    scores[doc_id] = score

ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

for doc_id, score in ranked[:3]:
    print(doc_id, round(score, 4))

# Now `alice` comes first: it is the only document containing the rare word
# "sister", and rarity is exactly what the idf weight rewards.